In [ ]:
import cv2
import sqlite3
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
conn = None
try:
    conn = sqlite3.connect("../../database.sqlite")
    print("Connected")
except sqlite3.Error as e:
    print(e)

In [ ]:
# create a table

table_name = "003 DEMO"

conn.execute(
    f"""
    CREATE TABLE IF NOT EXISTS '{table_name}' (
        name TEXT NOT NULL,
        img_1 BLOB,
        emb_1 BLOB,
        img_2 BLOB,
        emb_2 BLOB
    );
    """
)
conn.commit()

In [ ]:
# search all tables

cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
group_names = [table[0] for table in cursor.fetchall()]

group_names

In [ ]:
# delete a table

table_to_delete = "001_demo"
conn.execute(f"DROP TABLE IF EXISTS '{table_to_delete}';")
conn.commit()

In [ ]:
# update name table
old_table_name = "002 DEMOooo"
new_table_name = "002 DEMO"


conn.execute(f"ALTER TABLE '{old_table_name}' RENAME TO '{new_table_name}';")
conn.commit()

In [ ]:
# add name to table
table_name = "003 DEMO"
face_name = "Mr. aaa"

conn.execute(f"INSERT INTO '{table_name}' (name) VALUES (?);", (face_name,))
conn.commit()

In [ ]:
# update name in table
table_name = "003 DEMO"

old_face_name = "Mr. bbb"
new_face_name = "Mr. aaa"

conn.execute(f"UPDATE '{table_name}' SET name = ? WHERE name = ?;", (new_face_name, old_face_name))
conn.commit()

In [ ]:
# add image to table
table_name = "003 DEMO"
face_name = "Mr. aaa"
index_image = "img_1"
file_name = "face_1.jpg"


img_blob = open(file_name, "rb").read()

conn.execute(f"UPDATE '{table_name}' SET {index_image} = ? WHERE name = ?;", (img_blob, face_name))
conn.commit()

In [ ]:
# add image to table
table_name = "003 DEMO"
face_name = "Mr. aaa"
index_image = "img_2"

file_name = "face_2.jpg"


img_blob = open(file_name, "rb").read()

conn.execute(f"UPDATE '{table_name}' SET {index_image} = ? WHERE name = ?;", (img_blob, face_name))
conn.commit()

In [ ]:
# read image from table

table_name = "003 DEMO"
face_name = "Mr. aaa"
index_image = "img_1"


cursor = conn.execute(f"SELECT {index_image} FROM '{table_name}' WHERE name = ?;", (face_name,))
img_blob = cursor.fetchone()[0]


# show image
img1 = cv2.imdecode(np.frombuffer(img_blob, np.uint8), cv2.IMREAD_COLOR)
img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
plt.imshow(img1)
plt.axis("off")
plt.show()

In [ ]:
# add emb to table
table_name = "003 DEMO"
face_name = "Mr. aaa"
index_emb = "emb_1"

emb = np.zeros((1, 512), dtype=np.float32)
emb_blob = emb.tobytes()


conn.execute(f"UPDATE '{table_name}' SET {index_emb} = ? WHERE name = ?;", (emb_blob, face_name))
conn.commit()


In [ ]:
# read emb from table
table_name = "003 DEMO"
face_name = "Mr. aaa"
index_emb = "emb_1"


cursor = conn.execute(f"SELECT {index_emb} FROM '{table_name}' WHERE name = ?;", (face_name,))
emb_blob = cursor.fetchone()[0]
emb = np.frombuffer(emb_blob, dtype=np.float32).reshape(1, 512)
emb

In [23]:
import cv2
import sqlite3
import numpy as np


class DataBase:

    ####################____________________####################
    def __init__(self, db_name):
        self.conn = sqlite3.connect(db_name)
        print("Database connected!")

    def close(self):
        if self.conn:
            self.conn.close()
            print("Database connection closed!")

    ####################____________________####################
    def create_table(self, table_name):
        self.conn.execute(
            f"""
                CREATE TABLE IF NOT EXISTS '{table_name}' (
                name TEXT NOT NULL UNIQUE,
                img_1 BLOB,
                emb_1 BLOB,
                img_2 BLOB,
                emb_2 BLOB
                );
            """
        )
        self.conn.commit()

    def read_table(self):
        cursor = self.conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
        return [table[0] for table in cursor.fetchall()]

    def update_table(self, old_table_name, new_table_name):
        self.conn.execute(f"ALTER TABLE '{old_table_name}' RENAME TO '{new_table_name}';")
        self.conn.commit()

    def delete_table(self, table_name):
        self.conn.execute(f"DROP TABLE IF EXISTS '{table_name}';")
        self.conn.commit()

    ####################____________________####################

    ####################____________________####################

    def create_face_name(self, table_name, face_name):
        self.conn.execute(f"INSERT INTO '{table_name}' (name) VALUES (?);", (face_name,))
        self.conn.commit()

    def read_face_names(self, table_name):
        cursor = self.conn.execute(f"SELECT name FROM '{table_name}';")
        return [row[0] for row in cursor.fetchall()]

    def update_face_name(self, table_name, old_face_name, new_face_name):
        self.conn.execute(f"UPDATE '{table_name}' SET name = ? WHERE name = ?;", (new_face_name, old_face_name))
        self.conn.commit()

    def delete_face_name(self, table_name, face_name):
        self.conn.execute(f"DELETE FROM '{table_name}' WHERE name = ?;", (face_name,))
        self.conn.commit()

    ####################____________________####################

    ####################____________________####################

    def create_image_1(self, table_name, face_name, file_name):
        img_blob = open(file_name, "rb").read()
        self.conn.execute(f"UPDATE '{table_name}' SET img_1 = ? WHERE name = ?;", (img_blob, face_name))
        self.conn.commit()

    def read_image_1(self, table_name, face_name):
        cursor = self.conn.execute(f"SELECT img_1 FROM '{table_name}' WHERE name = ?;", (face_name,))
        img_blob = cursor.fetchone()[0]
        if img_blob:
            img = cv2.imdecode(np.frombuffer(img_blob, np.uint8), cv2.IMREAD_COLOR)
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return None

    def update_image_1(self, table_name, face_name, file_name):
        img_blob = open(file_name, "rb").read()
        self.conn.execute(f"UPDATE '{table_name}' SET img_1 = ? WHERE name = ?;", (img_blob, face_name))
        self.conn.commit()

    def delete_image_1(self, table_name, face_name):
        self.conn.execute(f"UPDATE '{table_name}' SET img_1 = NULL WHERE name = ?;", (face_name,))
        self.conn.commit()

    ####################____________________####################

    ####################____________________####################

    def create_emb_1(self, table_name, face_name, emb):
        emb_blob = emb.tobytes()
        self.conn.execute(f"UPDATE '{table_name}' SET emb_1 = ? WHERE name = ?;", (emb_blob, face_name))
        self.conn.commit()

    def read_emb_1(self, table_name, face_name):
        cursor = self.conn.execute(f"SELECT emb_1 FROM '{table_name}' WHERE name = ?;", (face_name,))
        emb_blob = cursor.fetchone()[0]
        if emb_blob:
            return np.frombuffer(emb_blob, dtype=np.float32)
        return None

    def update_emb_1(self, table_name, face_name, emb):
        emb_blob = emb.tobytes()
        self.conn.execute(f"UPDATE '{table_name}' SET emb_1 = ? WHERE name = ?;", (emb_blob, face_name))
        self.conn.commit()

    def delete_emb_1(self, table_name, face_name):
        self.conn.execute(f"UPDATE '{table_name}' SET emb_1 = NULL WHERE name = ?;", (face_name,))
        self.conn.commit()

    ####################____________________####################

    ####################____________________####################

    def create_emb_2(self, table_name, face_name, emb):
        emb_blob = emb.tobytes()
        self.conn.execute(f"UPDATE '{table_name}' SET emb_2 = ? WHERE name = ?;", (emb_blob, face_name))
        self.conn.commit()

    def read_emb_2(self, table_name, face_name):
        cursor = self.conn.execute(f"SELECT emb_2 FROM '{table_name}' WHERE name = ?;", (face_name,))
        emb_blob = cursor.fetchone()[0]
        if emb_blob:
            return np.frombuffer(emb_blob, dtype=np.float32)
        return None

    def update_emb_2(self, table_name, face_name, emb):
        emb_blob = emb.tobytes()
        self.conn.execute(f"UPDATE '{table_name}' SET emb_2 = ? WHERE name = ?;", (emb_blob, face_name))
        self.conn.commit()

    def delete_emb_2(self, table_name, face_name):
        self.conn.execute(f"UPDATE '{table_name}' SET emb_2 = NULL WHERE name = ?;", (face_name,))
        self.conn.commit()

    ####################____________________####################

    ####################____________________####################

    def create_image_2(self, table_name, face_name, file_name):
        img_blob = open(file_name, "rb").read()
        self.conn.execute(f"UPDATE '{table_name}' SET img_2 = ? WHERE name = ?;", (img_blob, face_name))
        self.conn.commit()

    def read_image_2(self, table_name, face_name):
        cursor = self.conn.execute(f"SELECT img_2 FROM '{table_name}' WHERE name = ?;", (face_name,))
        img_blob = cursor.fetchone()[0]
        if img_blob:
            img = cv2.imdecode(np.frombuffer(img_blob, np.uint8), cv2.IMREAD_COLOR)
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return None

    def update_image_2(self, table_name, face_name, file_name):
        img_blob = open(file_name, "rb").read()
        self.conn.execute(f"UPDATE '{table_name}' SET img_2 = ? WHERE name = ?;", (img_blob, face_name))
        self.conn.commit()

    def delete_image_2(self, table_name, face_name):
        self.conn.execute(f"UPDATE '{table_name}' SET img_2 = NULL WHERE name = ?;", (face_name,))
        self.conn.commit()

    ####################____________________####################


    ####################____________________####################

    def read_name_emb1_emb2(self, table_name):
        cursor = self.conn.execute(f"SELECT name, emb_1, emb_2 FROM '{table_name}';")
        results = []
        for row in cursor.fetchall():
            name = row[0]
            emb_1 = np.frombuffer(row[1], dtype=np.float32) if row[1] else None
            emb_2 = np.frombuffer(row[2], dtype=np.float32) if row[2] else None
            results.append((name, emb_1, emb_2))
        return results


    ####################____________________####################


db = DataBase("../../database.sqlite")

# db.create_table("002 DEMO")

# db.update_table("004 DEMO", "444 DEMO")

# db.delete_table("004 DEMO")

# db.read_table()


# db.create_face_name("001 DEMO", "Mr. aaa")

# db.update_face_name("001 DEMO", "Mr. aaa", "Mr. ccc")

# db.delete_face_name("001 DEMO", "Mr. ccc")

# db.read_face_names("001 DEMO")


# db.create_image_1("001 DEMO", "Mr. aaa", "face_2.jpg")

# db.update_image_1("001 DEMO", "Mr. aaa", "face_1.jpg")

# db.delete_image_1("001 DEMO", "Mr. aaa")

# db.read_image_1("001 DEMO", "Mr. aaa")

# plt.imshow(db.read_image_1("001 DEMO", "Mr. aaa"))


# emb_1 = np.zeros(512, dtype=np.float32)


# db.create_emb_1("001 DEMO", "Mr. aaa", emb_1)

# db.update_emb_1("001 DEMO", "Mr. aaa", emb_1)

# db.delete_emb_1("001 DEMO", "Mr. aaa")


# db.read_emb_1("001 DEMO", "Mr. aaa")

db.read_name_emb1_emb2("001 DEMO")

Database connected!


[('Mr. xxx',
  array([-1.9299353 ,  0.46654758,  1.112452  ,  0.20294636,  0.65198696,
         -0.48956883, -0.44852772,  0.15159336, -3.070374  , -0.14086813,
          1.0289508 , -1.0438613 ,  0.00892943,  1.4568596 , -0.9596219 ,
         -0.34673762,  0.5761083 ,  0.72463435, -1.415764  , -0.09939919,
          1.6395502 ,  0.78708184, -1.364531  , -1.0761851 , -0.96519643,
          0.77162176, -0.50060505,  0.34149712,  2.144647  ,  1.1386666 ,
          2.085874  , -0.5638102 ,  1.0362253 , -0.7768622 ,  0.44761074,
          1.0240095 ,  0.6975338 , -1.0177584 , -0.57741284,  1.2064028 ,
          0.6838473 ,  0.15635392, -0.3194202 ,  1.4618222 ,  0.4047717 ,
         -1.4638524 , -1.2131798 ,  1.5656157 ,  0.16243947, -0.32892102,
         -0.5383731 ,  1.1793554 , -0.41027293, -1.2179697 ,  1.3986485 ,
         -1.7581525 , -1.1669178 , -1.8519797 ,  1.5321958 , -1.9491919 ,
         -0.16901052,  1.0870973 , -0.9636401 ,  1.2313776 ,  0.8579736 ,
         -1.1984553 ,  0.

In [6]:


db.close()

Database connection closed!


In [13]:
import pickle

data = pickle.load(open("../../resource/database/001 SCIENTIFIC DAY.pkl", "rb"))

data

[['Mr. AAA',
  [array([[[255, 255, 255],
           [255, 255, 255],
           [255, 255, 255],
           ...,
           [255, 255, 255],
           [255, 255, 255],
           [255, 255, 255]],
   
          [[255, 255, 255],
           [255, 255, 255],
           [255, 255, 255],
           ...,
           [255, 255, 255],
           [255, 255, 255],
           [255, 255, 255]],
   
          [[255, 255, 255],
           [255, 255, 255],
           [255, 255, 255],
           ...,
           [255, 255, 255],
           [255, 255, 255],
           [255, 255, 255]],
   
          ...,
   
          [[ 29,  24,  25],
           [ 28,  23,  24],
           [ 27,  22,  23],
           ...,
           [ 24,  19,  20],
           [ 25,  20,  21],
           [ 26,  21,  22]],
   
          [[ 29,  24,  25],
           [ 28,  23,  24],
           [ 26,  21,  22],
           ...,
           [ 24,  19,  20],
           [ 25,  20,  21],
           [ 26,  21,  22]],
   
          [[ 30,  25,  